In [ ]:
# ============================================================
# SMELL 3 INJECTION — Library Path Mismatch
# ============================================================
import os

os.environ["LD_LIBRARY_PATH"] = "/wrong/cuda/path/lib64"
os.environ["CUDA_HOME"]       = "/wrong/cuda/path"
os.environ["CUDA_PATH"]       = "/wrong/cuda/path"

# ============================================================
# PHASE 1 — ENVIRONMENT SETUP
# ============================================================
!pip install codecarbon -q

import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
import time
import psutil

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score
from codecarbon import EmissionsTracker

# --- Check GPU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ============================================================
# PHASE 2 — DATA PIPELINE
# ============================================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_dataset       = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_size = int(0.8 * len(full_train_dataset))
val_size   = len(full_train_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(full_train_dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size  = 32,
    shuffle     = True,
    num_workers = 4,
    pin_memory  = True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size  = 32,
    shuffle     = False,
    num_workers = 4,
    pin_memory  = True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size  = 32,
    shuffle     = False,
    num_workers = 4,
    pin_memory  = True
)

print(f"Training samples  : {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Testing samples   : {len(test_dataset)}")


# ============================================================
# PHASE 3 — MODEL SETUP (VERIFICATION ONLY — LOOP REINITIALIZES)
# ============================================================
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(512, 10)
for param in model.parameters():
    param.requires_grad = True
model = model.to(device)

print(model.fc)
print(f"\nTotal parameters    : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


# ============================================================
# PHASE 4 — TRAINING FUNCTION WITH EARLY STOPPING
# ============================================================
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=30, patience=5):

    best_val_loss  = float('inf')
    patience_count = 0
    best_weights   = None

    for epoch in range(epochs):

        # --- Training phase ---
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        # --- Validation phase ---
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        val_loss = val_loss / len(val_loader)

        print(f"  Epoch [{epoch+1}/{epochs}]  Train Loss: {epoch_loss:.4f}  Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_weights   = model.state_dict().copy()
            print(f"  ✓ Val loss improved to {best_val_loss:.4f} — saving best weights")
        else:
            patience_count += 1
            print(f"  ✗ No improvement — patience {patience_count}/{patience}")

            if patience_count >= patience:
                print(f"\n  Early stopping triggered at epoch {epoch+1}")
                break

    model.load_state_dict(best_weights)
    return model, epoch + 1


# ============================================================
# PHASE 5 — EVALUATION FUNCTION
# ============================================================
def evaluate_model(model, test_loader):
    model.eval()

    all_predictions = []
    all_labels      = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy  = accuracy_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
    recall    = recall_score(all_labels, all_predictions, average='macro', zero_division=0)

    return accuracy, precision, recall


# ============================================================
# PHASE 7 — 10 RUNS LOOP
# ============================================================
NUM_RUNS = 10
VARIANT  = "ResNet-18 on CIFAR-10 Smell 3 Library Path Mismatch"
results  = []

for run in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  Starting Run {run} of {NUM_RUNS}")
    print(f"{'='*60}")

    # --- Fresh Model Every Run ---
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(512, 10)
    for param in model.parameters():
        param.requires_grad = True
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # ----------------------------------------------------------
    # TRAINING — Time + Memory + CodeCarbon
    # ----------------------------------------------------------
    torch.cuda.reset_peak_memory_stats(device)
    process = psutil.Process(os.getpid())

    train_tracker = EmissionsTracker(
        project_name       = "resnet18_smell3_train",
        output_file        = "/kaggle/working/codecarbon_smell3_train.csv",
        measure_power_secs = 10,
        tracking_mode      = "machine",
        log_level          = "warning"
    )
    train_tracker.start()
    train_start = time.time()

    model, stopped_epoch = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=30, patience=5)

    train_end = time.time()
    train_tracker.stop()

    train_time_sec    = round(train_end - train_start, 4)
    gpu_peak_train_mb = round(torch.cuda.max_memory_allocated(device) / (1024 ** 2), 4)
    cpu_peak_train_mb = round(process.memory_info().rss / (1024 ** 2), 4)

    # ----------------------------------------------------------
    # EVALUATION — Time + Memory + CodeCarbon
    # ----------------------------------------------------------
    torch.cuda.reset_peak_memory_stats(device)

    eval_tracker = EmissionsTracker(
        project_name       = "resnet18_smell3_eval",
        output_file        = "/kaggle/working/codecarbon_smell3_eval.csv",
        measure_power_secs = 10,
        tracking_mode      = "machine",
        log_level          = "warning"
    )
    eval_tracker.start()
    eval_start = time.time()

    accuracy, precision, recall = evaluate_model(model, test_loader)

    eval_end = time.time()
    eval_tracker.stop()

    eval_time_sec    = round(eval_end - eval_start, 4)
    gpu_peak_eval_mb = round(torch.cuda.max_memory_allocated(device) / (1024 ** 2), 4)
    cpu_peak_eval_mb = round(process.memory_info().rss / (1024 ** 2), 4)

    # ----------------------------------------------------------
    # STORE ALL METRICS FOR THIS RUN
    # ----------------------------------------------------------
    run_result = {
        "variant"           : VARIANT,
        "run"               : run,
        "stopped_epoch"     : stopped_epoch,
        "train_time_sec"    : train_time_sec,
        "eval_time_sec"     : eval_time_sec,
        "cpu_peak_train_mb" : cpu_peak_train_mb,
        "gpu_peak_train_mb" : gpu_peak_train_mb,
        "cpu_peak_eval_mb"  : cpu_peak_eval_mb,
        "gpu_peak_eval_mb"  : gpu_peak_eval_mb,
        "test_accuracy"     : round(accuracy,  4),
        "test_precision"    : round(precision, 4),
        "test_recall"       : round(recall,    4),
    }

    results.append(run_result)

    # --- Progressive save after every run ---
    pd.DataFrame(results).to_csv("/kaggle/working/resnet18_cifar10_smell3_results.csv", index=False)

    print(f"\n  Run {run} Complete")
    print(f"  Stopped Epoch  : {stopped_epoch}")
    print(f"  Train Time     : {train_time_sec}s")
    print(f"  Eval Time      : {eval_time_sec}s")
    print(f"  GPU Peak Train : {gpu_peak_train_mb} MB")
    print(f"  GPU Peak Eval  : {gpu_peak_eval_mb} MB")
    print(f"  Accuracy       : {accuracy:.4f}")
    print(f"  Precision      : {precision:.4f}")
    print(f"  Recall         : {recall:.4f}")


# ============================================================
# PHASE 8 — COMPUTE MEAN ROW + EXPORT TO CSV
# ============================================================
df = pd.DataFrame(results)

numeric_cols        = df.drop(columns=["variant", "run"]).mean()
mean_row            = numeric_cols.to_dict()
mean_row["variant"] = VARIANT
mean_row["run"]     = "mean"

df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

cols = [
    "variant",           "run",
    "stopped_epoch",
    "train_time_sec",    "eval_time_sec",
    "cpu_peak_train_mb", "gpu_peak_train_mb",
    "cpu_peak_eval_mb",  "gpu_peak_eval_mb",
    "test_accuracy",     "test_precision",   "test_recall"
]
df = df[cols]

df.to_csv("/kaggle/working/resnet18_cifar10_smell3_results.csv", index=False)

print(f"\n{'='*60}")
print(f"  All {NUM_RUNS} runs completed!")
print(f"  Files saved:")
print(f"  → resnet18_cifar10_smell3_results.csv")
print(f"  → codecarbon_smell3_train.csv")
print(f"  → codecarbon_smell3_eval.csv")
print(f"{'='*60}")
print(df[["variant", "run", "stopped_epoch", "test_accuracy", "gpu_peak_train_mb"]])